# Lesson 3: Streaming Responses & Chat Interfaces

In this lesson, you'll learn how to build responsive, real-time AI applications using streaming.

## Topics Covered
1. Understanding streaming vs non-streaming responses
2. Implementing streaming for real-time output
3. Building interactive chat interfaces
4. Handling partial responses and streaming errors
5. Managing conversation history

## Learning Objectives
- Implement streaming responses for better UX
- Build a functional chat interface
- Handle errors gracefully in streaming mode
- Manage multi-turn conversations with context

In [ ]:
# Install required packages if not already installed
#%pip install python-dotenv
#%pip install openai

# Load environment variables from .env file
import os
from dotenv import load_dotenv
load_dotenv()
import openai
import time
from datetime import datetime
print("OpenAI package version:", openai.__version__)

In [ ]:
# Set up the OpenAI client with environment variables
chat_client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=int(os.getenv("OPENAI_TIMEOUT", 30)),
    max_retries=int(os.getenv("MAX_RETRIES", 3)),
    base_url=os.getenv("OPENAI_ENDPOINT")    
)

print("Client configured successfully!")

## 1. Streaming vs Non-Streaming Responses

**Non-Streaming (Lesson 1 & 2):**
- Wait for complete response
- All-or-nothing approach
- User sees nothing until finished

**Streaming:**
- Tokens arrive in real-time
- Better user experience (like ChatGPT)
- Can process partial responses
- Feels more responsive

In [ ]:
# Example 1A: Non-streaming (what we've been doing)
print("Non-Streaming Response:")
print("Waiting for complete response...\n")

start_time = time.time()
response = chat_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "Write a 3-sentence story about a time traveler."}
    ]
)
elapsed_time = time.time() - start_time

print(response.choices[0].message.content)
print(f"\n⏱️  Total time: {elapsed_time:.2f}s")
print("\n" + "="*80 + "\n")

In [ ]:
# Example 1B: Streaming response
print("Streaming Response:")
print("Response appears in real-time:\n")

start_time = time.time()
stream = chat_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "Write a 3-sentence story about a time traveler."}
    ],
    stream=True  # Enable streaming!
)

# Process each chunk as it arrives
full_response = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        content = chunk.choices[0].delta.content
        full_response += content
        print(content, end='', flush=True)  # Print immediately

elapsed_time = time.time() - start_time
print(f"\n\n⏱️  Total time: {elapsed_time:.2f}s")
print("(Notice: First words appear much faster!)")

## 2. Implementing Streaming for Real-Time Output

Let's build reusable streaming functions with better control.

In [ ]:
# Example 2A: Basic streaming function
def stream_response(messages, show_chunks=False):
    """Stream a response and return the complete text"""
    stream = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=messages,
        stream=True
    )
    
    full_response = ""
    chunk_count = 0
    
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            full_response += content
            chunk_count += 1
            
            if show_chunks:
                print(f"[Chunk {chunk_count}]: {repr(content)}")
            else:
                print(content, end='', flush=True)
    
    print()  # New line at end
    return full_response

# Test it
print("Testing basic streaming function:\n")
result = stream_response([
    {"role": "user", "content": "Explain what an API is in one sentence."}
])
print(f"\nCaptured: {result}")

In [ ]:
# Example 2B: See the chunks (for debugging)
print("Showing individual chunks:\n")
result = stream_response(
    [{"role": "user", "content": "Count: 1, 2, 3, 4, 5"}],
    show_chunks=True
)
print(f"\nNotice: Tokens don't always align with words!")

In [ ]:
# Example 2C: Streaming with metadata
def stream_with_metadata(messages):
    """Stream response and capture metadata"""
    stream = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=messages,
        stream=True
    )
    
    full_response = ""
    first_chunk_time = None
    start_time = time.time()
    chunk_count = 0
    finish_reason = None
    
    for chunk in stream:
        chunk_count += 1
        
        # Capture timing of first chunk
        if first_chunk_time is None:
            first_chunk_time = time.time() - start_time
        
        # Get content
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            full_response += content
            print(content, end='', flush=True)
        
        # Capture finish reason
        if chunk.choices[0].finish_reason is not None:
            finish_reason = chunk.choices[0].finish_reason
    
    total_time = time.time() - start_time
    
    print()  # New line
    return {
        "content": full_response,
        "chunk_count": chunk_count,
        "first_chunk_time": first_chunk_time,
        "total_time": total_time,
        "finish_reason": finish_reason
    }

# Test it
print("Streaming with metadata:\n")
result = stream_with_metadata([
    {"role": "user", "content": "List 5 programming languages."}
])

print(f"\n📊 Metadata:")
print(f"  - First chunk: {result['first_chunk_time']:.3f}s")
print(f"  - Total time: {result['total_time']:.2f}s")
print(f"  - Chunks received: {result['chunk_count']}")
print(f"  - Finish reason: {result['finish_reason']}")

## 3. Building Interactive Chat Interfaces

Let's create a simple but functional chat system with conversation history.

In [ ]:
# Example 3A: Simple chat class
class SimpleChat:
    def __init__(self, system_message="You are a helpful assistant."):
        self.messages = [
            {"role": "system", "content": system_message}
        ]
    
    def send_message(self, user_message, stream=True):
        """Send a message and get streaming response"""
        # Add user message to history
        self.messages.append({"role": "user", "content": user_message})
        
        print(f"\n🧑 You: {user_message}")
        print("🤖 Assistant: ", end='', flush=True)
        
        # Get streaming response
        stream_obj = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=self.messages,
            stream=stream
        )
        
        full_response = ""
        for chunk in stream_obj:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                full_response += content
                print(content, end='', flush=True)
        
        print()  # New line
        
        # Add assistant response to history
        self.messages.append({"role": "assistant", "content": full_response})
        
        return full_response
    
    def get_history(self):
        """Return conversation history (excluding system message)"""
        return self.messages[1:]  # Skip system message
    
    def clear_history(self):
        """Clear conversation history (keep system message)"""
        self.messages = self.messages[:1]

# Test the chat
print("=" * 80)
print("Testing Simple Chat Interface")
print("=" * 80)

chat = SimpleChat(system_message="You are a friendly Python tutor.")
chat.send_message("What is a list in Python?")
chat.send_message("Can you show me an example?")

print(f"\n📝 Total messages in history: {len(chat.get_history())}")

In [ ]:
# Example 3B: Enhanced chat with features
class EnhancedChat:
    def __init__(self, system_message="You are a helpful assistant.", max_history=10):
        self.system_message = system_message
        self.max_history = max_history
        self.messages = [{"role": "system", "content": system_message}]
        self.message_count = 0
    
    def send_message(self, user_message, stream=True, temperature=0.7):
        """Send a message with enhanced features"""
        self.message_count += 1
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        # Add user message
        self.messages.append({"role": "user", "content": user_message})
        
        print(f"\n[{timestamp}] 🧑 You: {user_message}")
        print(f"[{timestamp}] 🤖 Assistant: ", end='', flush=True)
        
        # Get response
        stream_obj = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=self.messages,
            stream=stream,
            temperature=temperature
        )
        
        full_response = ""
        for chunk in stream_obj:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                full_response += content
                print(content, end='', flush=True)
        
        print()
        
        # Add assistant response
        self.messages.append({"role": "assistant", "content": full_response})
        
        # Trim history if too long (keep system message + recent messages)
        if len(self.messages) > self.max_history + 1:
            self.messages = [self.messages[0]] + self.messages[-(self.max_history):]
            print(f"\n⚠️  History trimmed to last {self.max_history} messages")
        
        return full_response
    
    def display_history(self):
        """Pretty print conversation history"""
        print("\n" + "=" * 80)
        print("📜 CONVERSATION HISTORY")
        print("=" * 80)
        for msg in self.messages[1:]:  # Skip system message
            role = "🧑 You" if msg["role"] == "user" else "🤖 Assistant"
            content = msg["content"][:100] + "..." if len(msg["content"]) > 100 else msg["content"]
            print(f"{role}: {content}\n")
    
    def get_stats(self):
        """Get conversation statistics"""
        user_msgs = [m for m in self.messages if m["role"] == "user"]
        assistant_msgs = [m for m in self.messages if m["role"] == "assistant"]
        
        return {
            "total_messages": len(self.messages) - 1,  # Exclude system
            "user_messages": len(user_msgs),
            "assistant_messages": len(assistant_msgs),
            "total_turns": self.message_count
        }

# Test enhanced chat
print("=" * 80)
print("Testing Enhanced Chat Interface")
print("=" * 80)

chat = EnhancedChat(
    system_message="You are a helpful coding assistant. Keep responses concise.",
    max_history=6
)

chat.send_message("What's the difference between a list and tuple in Python?")
chat.send_message("Which one should I use for storing coordinates?")

# Show stats
stats = chat.get_stats()
print(f"\n📊 Stats: {stats['total_turns']} turns, {stats['total_messages']} messages")

## 4. Handling Partial Responses and Errors

Streaming can fail mid-response. Let's handle errors gracefully.

In [ ]:
# Example 4A: Basic error handling
def safe_stream(messages, max_retries=3):
    """Stream with error handling and retries"""
    for attempt in range(max_retries):
        try:
            stream = chat_client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
                messages=messages,
                stream=True
            )
            
            full_response = ""
            for chunk in stream:
                if chunk.choices[0].delta.content is not None:
                    content = chunk.choices[0].delta.content
                    full_response += content
                    print(content, end='', flush=True)
            
            print()  # New line
            return full_response
            
        except openai.APIError as e:
            print(f"\n❌ API Error on attempt {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"⏳ Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print("❌ Max retries reached. Giving up.")
                return None
        
        except Exception as e:
            print(f"\n❌ Unexpected error: {e}")
            return None

# Test it
print("Testing error handling:\n")
result = safe_stream([
    {"role": "user", "content": "What is machine learning?"}
])

if result:
    print("\n✅ Success!")
else:
    print("\n❌ Failed to get response")

In [ ]:
# Example 4B: Handling incomplete responses
def stream_with_completion_check(messages):
    """Stream and check if response completed"""
    try:
        stream = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=messages,
            stream=True,
            max_tokens=50  # Artificially low to show truncation
        )
        
        full_response = ""
        finish_reason = None
        
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                full_response += content
                print(content, end='', flush=True)
            
            if chunk.choices[0].finish_reason is not None:
                finish_reason = chunk.choices[0].finish_reason
        
        print()  # New line
        
        # Check completion status
        if finish_reason == "stop":
            print("\n✅ Response completed naturally")
        elif finish_reason == "length":
            print("\n⚠️  Response truncated (hit token limit)")
        elif finish_reason == "content_filter":
            print("\n⚠️  Response filtered for safety")
        else:
            print(f"\n⚠️  Response ended: {finish_reason}")
        
        return {
            "content": full_response,
            "finish_reason": finish_reason,
            "complete": finish_reason == "stop"
        }
    
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return None

# Test with token limit
print("Testing completion check (with artificial token limit):\n")
result = stream_with_completion_check([
    {"role": "user", "content": "Write a detailed explanation of neural networks."}
])

if result:
    print(f"\nResponse complete: {result['complete']}")
    print(f"Finish reason: {result['finish_reason']}")